Load the trained model

In [21]:
import cv2 as cv
import torch
import torch.nn as nn
import numpy as np


class MathSymbolCNN(nn.Module):
    def __init__(self, num_classes=47):
        super(MathSymbolCNN, self).__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),     # 28 x 28 -> 14 x 14

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)      # 14 x 14 -> 7 x 7
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x
    
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MathSymbolCNN().to(device)
model.load_state_dict(torch.load('math_symbol_cnn.pth', map_location=device))
model.eval()

MathSymbolCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.5, inplace=False)
    (4): Linear(in_features=256, out_features=47, bias=True)
  )
)

Import dataset

In [22]:
import torchvision
dataset = torchvision.datasets.EMNIST(root='./data', split='balanced', train=False, download=False)
classes = dataset.classes

Image segmentation (extract symbols)

In [23]:
def segment_image(image_path):
    img = cv.imread(image_path)
    gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
    blurred = cv.GaussianBlur(gray, (3, 3), 0)
    thresh = cv.adaptiveThreshold(
        blurred, 255, cv.ADAPTIVE_THRESH_GAUSSIAN_C, cv.THRESH_BINARY_INV, 15, 8
    )
    kernel = cv.getStructuringElement(cv.MORPH_RECT, (3,3))
    dilated = cv.dilate(thresh, kernel, iterations=1)
    contours, hierarchy = cv.findContours(dilated, cv.RETR_CCOMP, cv.CHAIN_APPROX_SIMPLE)

    image_area = gray.shape[0] * gray.shape[1]
    min_area = image_area * 0.001
    max_area = image_area * 0.95

    filtered = []
    for i, contour in enumerate(contours):
        area = cv.contourArea(contour)
        has_no_parent = hierarchy[0][i][3] == -1
        if has_no_parent and min_area < area < max_area:
            filtered.append(contour)

    filtered = sorted(filtered, key=lambda x: cv.boundingRect(x)[0])

    crops = []
    SYMBOL_SIZE = 28
    PADDING = 4
    for contour in filtered:
        x, y, w, h = cv.boundingRect(contour)

        #Add paddings
        x1 = max(0, x - PADDING)
        y1 = max(0, y - PADDING)
        x2 = min(thresh.shape[1], x + w + PADDING)
        y2 = min(thresh.shape[0], y + h + PADDING)

        crop = thresh[y1:y2, x1:x2]

        crop_resized = cv.resize(crop, (SYMBOL_SIZE, SYMBOL_SIZE))
        crops.append(crop_resized)

    return crops, img

Recognition

In [24]:
def recognise_symbols(crops):
    recognised = []
    confidences = []

    for crop in crops:
        crop = cv.rotate(crop, cv.ROTATE_90_CLOCKWISE)  #EMNIST dataset is rotated and flipped
        crop = cv.flip(crop, 1)
        tensor = torch.tensor(crop, dtype=torch.float32) / 255.0
        tensor = tensor.unsqueeze(0).unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(tensor)
            probabilities = torch.softmax(output, dim=1)
            confidence, predicted = torch.max(probabilities, 1)

        recognised.append(classes[predicted.item()])
        confidences.append(confidence.item())

    return recognised, confidences

Main

In [30]:
def recognise_image(image_path):
    crops, original_image = segment_image(image_path)
    recognised, confidences = recognise_symbols(crops)

    print(f"Detected {len(recognised)} symbols")

    for i, (char, conf) in enumerate(zip(recognised, confidences)):
        print(f"Symbol {i + 1}: {char}  Confidence: {conf}")

    print(f"Recognised expression: {''.join(recognised)}")

    return recognised, confidences

recognise_image('image2.png')

Detected 7 symbols
Symbol 1: Y  Confidence: 0.9985429048538208
Symbol 2: r  Confidence: 0.8303180932998657
Symbol 3: M  Confidence: 0.47055086493492126
Symbol 4: 3  Confidence: 0.9976939558982849
Symbol 5: X  Confidence: 0.9994926452636719
Symbol 6: r  Confidence: 0.6460641622543335
Symbol 7: I  Confidence: 0.4082343280315399
Recognised expression: YrM3XrI


(['Y', 'r', 'M', '3', 'X', 'r', 'I'],
 [0.9985429048538208,
  0.8303180932998657,
  0.47055086493492126,
  0.9976939558982849,
  0.9994926452636719,
  0.6460641622543335,
  0.4082343280315399])